# 1 · Simulate spots and noise
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
DECODE simulates emitters using a spline PSF, background, shot noise and camera response. Here we use the bundled **example EMCCD parameters and PSF**. Their values do not describe the lab Nanoimager. We simulate a small number of frames, not a training run.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Change this on the lab PC. It must contain experimental_data_workflow/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'experimental_data_workflow').exists() and (ROOT.parent / 'experimental_data_workflow').exists():
    ROOT = ROOT.parent
EXAMPLE = ROOT / 'experimental_data_workflow'
WORK = ROOT / 'decode_work'
print('Project:', ROOT, '| Example exists:', EXAMPLE.exists())


In [ ]:
import torch
import decode
import decode.utils.param_io
from decode.neuralfitter.train.random_simulation import setup_random_simulation
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


### Set example simulation parameters
`emitter_av` is the average number of emitters in each 40 × 40 frame. Background and brightness are photons under the model convention. The camera adds noise; `convert2photons=True` converts the simulated camera output back into estimated photon units. Change values and rerun. Keep the example camera parameters unchanged for the example PSF; real sCMOS values must be measured.


In [ ]:
param = decode.utils.param_io.load_params(str(EXAMPLE / 'param_run.yaml'))
param.InOut.calibration_file = str(EXAMPLE / 'spline_calibration_3dcal.mat')
param.Hardware.device_simulation = 'cpu'  # try 'cuda:0' only if torch AND spline support CUDA
param.Hardware.device = 'cpu'
param.Simulation.emitter_av = 25
param.Simulation.bg_uniform = [20.0, 200.0]
param.Simulation.intensity_mu_sig = [7000.0, 3000.0]
param.HyperParameter.pseudo_ds_size = 7  # inclusive frame range 0..7
param.TestSet.test_size = 7
torch.manual_seed(42)
sim, _ = setup_random_simulation(param)
emitters, frames, backgrounds = sim.sample()
frames = frames.detach().cpu().numpy()
print('Simulated frames:', frames.shape, '| convert to photons:', param.Camera.convert2photons)
plt.figure(figsize=(6,5)); plt.imshow(frames[0], cmap='gray'); plt.colorbar(label='model output'); plt.title('DECODE example simulation'); plt.show()


### Compare both raw channels with the same display scale
Choose your own TIFF; this cell reads only one frame. The example simulation is in photon units, but your raw TIFF is in camera ADU; do not compare numerical values until you have a validated camera conversion. Auto-contrast can make a background look white.


In [ ]:
RAW_TIFF = None  # e.g. ROOT / 'high_density_examples' / 'recording.tif'
FRAME = 0
if RAW_TIFF is None:
    print('Set RAW_TIFF to compare the left and right halves.')
else:
    with tifffile.TiffFile(RAW_TIFF) as tif:
        raw = tif.pages[FRAME].asarray()
    if raw.ndim != 2 or raw.shape[1] % 2: raise ValueError('Expected a 2D frame with two equal-width channels.')
    left, right = np.split(raw, 2, axis=1)
    vmin, vmax = np.percentile(raw, [1, 99.9]) # same limits for both; set fixed numbers to compare different movies
    fig, axes = plt.subplots(1, 2, figsize=(12,5))
    for ax, img, name in zip(axes, [left, right], ['Left', 'Right']):
        im = ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax); ax.set_title(name); print(name, 'ADU percentiles:', np.percentile(img, [1,50,99,99.9]))
    fig.colorbar(im, ax=axes, label='ADU'); plt.show()
